This is a notebook to compare the source STL shapes that we'll use to make the 3D printed objects to the scanned point clouds from the Intel Real Sense cameras. 

In [1]:
# imports and path configuration
import sys
import os
import open3d as o3d
import numpy as np

# Add the 'Experimental' directory to the system path so we can import from '3d scan'
# Note: Since '3d scan' has a space, we use a slightly different approach to import it
sys.path.append(os.path.abspath("Experimental"))

# We use importlib to handle the folder name with a space
import importlib.util
module_name = "three_d_scan" # We treat it as a module
file_path = os.path.join("3d scan", "process_3d_files.py")

# Load the module manually to bypass the space in the folder name
spec = importlib.util.spec_from_file_location(module_name, file_path)
p3f = importlib.util.module_from_spec(spec)
spec.loader.exec_module(p3f)

# import script used to generate the actual
gen_file_path = os.path.abspath(r"D:\Projects\3D-textures\Generator\Generator.py")

# We use importlib to handle the import, same as we did for the 3d scan folder
spec = importlib.util.spec_from_file_location("Generator", gen_file_path)
gen_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(gen_module)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
# ==========================
# Processing
# ==========================

# path to a point cloud
ply_path1 = r"D:\Projects\3D-textures\processed_models\BAMBU\B1T2.ply"

# Call the function
# This will return the PointCloud object as a variable
ply_obj1 = p3f.process_ply(ply_path1, rm_floor=True)

if ply_obj1 is not None:
    print("Successfully loaded and processed point cloud.")
    
    # ==========================
    # Visualization
    # ==========================
    # This will open a separate interactive window showing the 3D point cloud
    #o3d.visualization.draw_geometries([ply_obj1], window_name="3D Scan Result")
else:
    print("Failed to process the point cloud. Check if the path is correct.")

Processed D:\Projects\3D-textures\processed_models\BAMBU\B1T2.ply (no file saved)
Successfully loaded and processed point cloud.


In [2]:
# ==========================
# Generate and Visualize z1 Surface
# ==========================

# Grab the math variables directly from the imported Generator module
# These were defined and calculated when the module was loaded
x_data = gen_module.x
y_data = gen_module.y
z1_data = gen_module.z1

# Use the modified surface_to_stl function to get a PointCloud object
# We pass filename=None so it returns the object instead of saving to disk
pcd_z1 = gen_module.surface_to_stl(x_data, y_data, z1_data, filename=None)

if pcd_z1 is not None:
    print(f"Generated PointCloud for z1. Number of points: {len(pcd_z1.points)}")
    
    # Visualize the result
    # This will open the Open3D window
    #o3d.visualization.draw_geometries([pcd_z1], window_name="Generator Texture: z1")
else:
    print("Failed to generate PointCloud for z1.")

Returned PointCloud (no file saved)
Generated PointCloud for z1. Number of points: 88494


In [5]:
# 1. Load the reference (STL)
# This returns an object directly
ref_obj = gen_module.surface_to_stl(gen_module.x, gen_module.y, gen_module.z1, filename=None)

# 2. Load and process the scan (PLY)
# path to a point cloud
ply_path1 = r"D:\Projects\3D-textures\processed_models\BAMBU\B1T2.ply"
# This returns an object directly
scan_obj = p3f.process_ply(ply_path1, rm_floor=True)

if ref_obj is not None and scan_obj is not None:
    # 3. Align the two objects directly
    # We set rm_floor=False here because process_ply already handled it
    aligned_ref, aligned_scan = p3f.align_ply_objects(ref_obj, scan_obj, rm_floor=False, scale=True)
    
    if aligned_ref is not None:
        # 4. Visualize
        o3d.visualization.draw_geometries([aligned_ref, aligned_scan], window_name="Surface Overlay")
        
        # 5. Calculate
        avg, std = p3f.calc(aligned_ref, aligned_scan)
        print(f"Average: {avg}, STD: {std}")

Returned PointCloud (no file saved)
Processed D:\Projects\3D-textures\processed_models\BAMBU\B1T2.ply (no file saved)
Detected Scale Factor: 937.4590x
Average: 8.139995328103694, STD: 5.916295378396873


In [10]:
#o3d.visualization.draw_geometries([aligned_scan], window_name="Surface Overlay")
#o3d.visualization.draw_geometries([aligned_ref, aligned_scan], window_name="Surface Overlay")

o3d.visualization.draw_geometries([ref_obj, scan_obj], window_name="Surface Overlay")

In [3]:
#Do complete loading and comparison process
# load ref
ref_obj = gen_module.surface_to_stl(gen_module.x, gen_module.y, gen_module.z1, filename=None)
# load scan
ply_path1 = r"D:\Projects\3D-textures\processed_models\BAMBU\B1T2.ply"
scan_obj = p3f.process_ply(ply_path1, rm_floor=True)

# --- Visualisation 1: Raw comparison ---
# Paint them different colors so you can see the overlap/offset clearly
ref_obj.paint_uniform_color([1, 0, 0])  # Red
scan_obj.paint_uniform_color([0, 1, 0])  # Green
print("Showing Raw Comparison (Red: STL, Green: Scan)...")
o3d.visualization.draw_geometries([ref_obj, scan_obj])

# --- Next scale the scan according to the ref ---
# We use your helper function to fix the size difference
scaled_scan_obj = p3f.scale_pcd_to_reference(ref_obj, scan_obj)

# --- Visualisation 2: After scaling ---
# The green cloud should now be roughly the same size as the red cloud
scaled_scan_obj.paint_uniform_color([0, 1, 0]) 
print("Showing Scaled Comparison...")
o3d.visualization.draw_geometries([ref_obj, scaled_scan_obj])

# --- Next perform alignment ---
# We set rm_floor=False and scale=False because those steps are already complete
# align_ply_objects returns (ref_obj, aligned_scan)
ref_obj, aligned_scan = p3f.align_ply_objects(ref_obj, scaled_scan_obj, rm_floor=False, scale=False)

# --- Visualisation 3: Final Alignment ---
# The green cloud should now be "snapped" onto the red cloud
aligned_scan.paint_uniform_color([0, 1, 0])
print("Showing Final Alignment...")
o3d.visualization.draw_geometries([ref_obj, aligned_scan])

# Final step: Calculate metrics
avg, std = p3f.calc(ref_obj, aligned_scan)
print(f"Final Metrics -> Average Deviation: {avg:.4f}, STD: {std:.4f}")

Returned PointCloud (no file saved)
Processed D:\Projects\3D-textures\processed_models\BAMBU\B1T2.ply (no file saved)
Showing Raw Comparison (Red: STL, Green: Scan)...
Detected Scale Factor: 937.5310x
Showing Scaled Comparison...
Showing Final Alignment...
Final Metrics -> Average Deviation: 8.1261, STD: 5.9121


In [10]:
# This function is from the "align_ply_objects" function from the original file process_3d_files.py, but I'm editing it to fix the alignment. 
def align_stlpcd_pcd(stl_pcd, pcd, rm_floor=False, scale=False):
    """
    Aligns two existing PointCloud objects using Point-to-Point ICP.
    
    Args:
        obj1 (o3d.geometry.PointCloud): Point cloud from stl.
        obj2 (o3d.geometry.PointCloud): Point cloud from scan.
        rm_floor (bool): Whether to run the floor removal logic on the scan pointcloud object.
            Defaults to False.

    Returns:
        tuple: (obj1, obj2) where obj2 is aligned to obj1, or None if failed.
    """
    try:
        # Check if inputs are valid point clouds
        if stl_pcd is None or pcd is None:
            print("Error: One or both input objects are None.")
            return None
        
        if len(stl_pcd.points) == 0 or len(pcd.points) == 0:
            print(f"Error: One of the objects is empty. STL: {len(stl_pcd.points)}, PCD: {len(pcd.points)}")
            return None

        # --- CENTERING STEP ---
        # This eliminates the "shift" by moving both centers to (0,0,0)
        stl_center = stl_pcd.get_center()
        pcd_center = pcd.get_center()
        
        # Translate both so they are centered at the origin
        stl_pcd.translate(-stl_center)
        pcd.translate(-pcd_center)
        # ----------------------------

        # Clean floor elements if requested
        if rm_floor:
            pcd = p3f.remove_floor(pcd)

        # Scale pcd to the scale of stl_pcd
        if scale:
            pcd = p3f.scale_pcd_to_reference(stl_pcd, pcd)

        # Base downsample settings
        voxel_size = 0.005

        # Multi-stage gates: Fix coarse angle tweaks, then clamp down tight.
        stages = [
            {"voxel": voxel_size * 2, "thresh": 0.040},  # Loose search gate
            {"voxel": voxel_size * 1, "thresh": 0.015},  # Medium search gate
            {"voxel": voxel_size * 0.5, "thresh": 0.004} # Tight sub-millimetre lock
        ]

        # Natural close start position means initial guess is an Identity Matrix
        current_transformation = np.identity(4)

        for i, stage in enumerate(stages):
            s_stl_pcd = stl_pcd.voxel_down_sample(stage["voxel"])
            s_pcd = pcd.voxel_down_sample(stage["voxel"])
            
            # PointToPoint avoids plane-sliding or rotation drift on flatter geometries
            result = o3d.pipelines.registration.registration_icp(
                source=s_pcd,
                target=s_stl_pcd,
                max_correspondence_distance=stage["thresh"],
                init=current_transformation,
                estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False)
            )
            current_transformation = result.transformation

        # Apply alignment matrix to original full-resolution source cloud (obj2)
        pcd.transform(current_transformation)

        # Colorize for explicit visual confirmation
        stl_pcd.paint_uniform_color([1, 0, 0])  # Red (Target reference)
        pcd.paint_uniform_color([0, 1, 0])  # Green (Aligned source)
        
        return stl_pcd, pcd

    except Exception as e:
        print(f"Alignment execution failed: {e}")
        return None


In [12]:
ref_obj_a, scan_obj_a = align_stlpcd_pcd(ref_obj, scan_obj)
o3d.visualization.draw_geometries([ref_obj_a, scan_obj_a])
avg, std = p3f.calc(ref_obj_a, scan_obj_a)
print(f"Final Metrics -> Average Deviation: {avg:.4f}, STD: {std:.4f}")

Final Metrics -> Average Deviation: 0.8257, STD: 0.4591
